In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model = init_chat_model('deepseek-chat')

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [2]:
# Step 1: Model generates tool calls
messages = [HumanMessage("What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)  # type: ignore
ai_msg.pretty_print()

================================== Ai Message ==================================

I'll check the weather in Boston for you.
Tool Calls:
  get_weather (call_00_t8mKmwkRKEGgMGxd0NwsDi3c)
 Call ID: call_00_t8mKmwkRKEGgMGxd0NwsDi3c
  Args:
    location: Boston


In [5]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'call_00_t8mKmwkRKEGgMGxd0NwsDi3c',
  'type': 'tool_call'}]

In [6]:
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    tool_result.pretty_print()

================================= Tool Message =================================
Name: get_weather

It's sunny in Boston.


In [7]:
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
final_response.pretty_print()

================================== Ai Message ==================================

The weather in Boston is sunny!
